# **Extracción y Análisis Exploratorio (EDA): Dinastía de Bajo Costo**

| *Bruno Gael Ramos Huerta*

**¿Qué buscamos en los datos?**
Demostrar que el equipo optó por la imprevisibilidad táctica en lugar del gasto excesivo en una sola posición, para esto analizaremos dos eras:

* **La Era Hill (2018 - 2021):** Años de alta dependencia a dos jugadores, entre ellos Hill.

* **La Era Post-Hill (2022 - 2023):** Temporadas del bicampeonato donde el esquema mutó hacia una mayor distribución del balón.

Utilizaremos los datos oficiales de jugada por jugada (*Play-by-Play*) para calcular la eficiencia ofensiva mediante el **EPA (Expected Points Added)** y medir cómo varió la distribución de los pases en el campo de juego.

### **Librerías**

In [4]:
import pandas as pd
import nfl_data_py as nfl

### **Carga de datos**
Elegimos datos desde 2018 hasta el año 2023. Y revisamos las dimensiones del dataset.

In [5]:
years = [2018, 2019, 2020, 2021, 2022, 2023]
df_pbp_raw = nfl.import_pbp_data(years)

2018 done.
2019 done.
2020 done.
2021 done.
2022 done.
2023 done.
Downcasting floats.


Este dataset registra cada mínimo detalle del partido (clima, tiempos fuera, castigos, posición de los árbitros, etc.). Haremos una exploración inicial para encontrar únicamente las varibles que necesitemos.

In [6]:
df_pbp_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291095 entries, 0 to 291094
Columns: 396 entries, play_id to defense_numbers
dtypes: float32(205), int32(7), int64(1), object(183)
memory usage: 644.1+ MB


In [7]:
df_pbp_raw.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,1.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,None,None,None,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
1,37.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
2,52.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
3,75.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,False,HITCH,ZONE_COVERAGE,COVER_3,NaN,NaN,NaN,NaN,NaN,NaN
4,104.0,2018_01_ATL_PHI,2018090600,PHI,ATL,REG,1,ATL,away,PHI,...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN


Tenemos casi 400 columnas dentro de nuestro dataset y más de 290,000 filas de datos, habría que hacer un filtrado con los datos que realmente nos interesan.

## **1. Análisis Exploratorio y Filtrado de Variables** 

Para nuestra narrativa, no necesitamos todas estas columnas que vienen dentro del dataset. Nos centraremos principalmente en estadísticas de juego aéreo, jugadores y métricas relevantes en cada partido.

#### **¿Por qué enfocarnos en el juego aéreo?**

Antes de analizar la reestructuración de Andy Reid, es importante justificar por qué decidimos excluir el juego terrestre de nuestra ecuación. Por ello, conservaremos temporalmente las jugadas etiquetadas como `run` (acarreos) además de las de `pass` (pases). Esto nos permitirá construir una visualización inicial que compare el EPA generado por aire vs el de tierra, demostrando la superioridad del pase en la NFL moderna.

Extraeremos 15 columnas críticas que nos permiten identificar a los equipos, a los jugadores involucrados, la distancia de la jugada y su valor en EPA.

In [8]:
df_plays = df_pbp_raw[
    (df_pbp_raw['play_type'].isin(['pass', 'run'])) & (df_pbp_raw['qb_kneel'] == 0)
    ].copy()

# Estas son las columnas críticas que seleccionamos
columnas_narrativa = [
    'game_id', 'play_id', 'season', 'posteam', 'defteam', 'play_type',
    'passer_player_name', 'receiver_player_name', 'rusher_player_name',
    'yards_gained', 'air_yards', 'epa', 'complete_pass', 'touchdown', 'pass_location'
]

df_plays_clean = df_plays[columnas_narrativa]
df_plays_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 207839 entries, 3 to 291093
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   game_id               207839 non-null  object 
 1   play_id               207839 non-null  float32
 2   season                207839 non-null  int64  
 3   posteam               207839 non-null  object 
 4   defteam               207839 non-null  object 
 5   play_type             207839 non-null  object 
 6   passer_player_name    122043 non-null  object 
 7   receiver_player_name  110086 non-null  object 
 8   rusher_player_name    85796 non-null   object 
 9   yards_gained          207839 non-null  float32
 10  air_yards             113481 non-null  float32
 11  epa                   207838 non-null  float32
 12  complete_pass         207839 non-null  float32
 13  touchdown             207839 non-null  float32
 14  pass_location         113475 non-null  object 
dtypes: fl

Después del filtrado, conservamos **207,839 jugadas clave** y **15 variables**. La idea es posteriormente utilizar este dataframe, limpiarlo y analizar los valores nulos que existen en distintas columnas.

In [9]:
df_plays_clean.head()

,game_id,play_id,season,posteam,defteam,play_type,passer_player_name,receiver_player_name,rusher_player_name,yards_gained,air_yards,epa,complete_pass,touchdown,pass_location
3,2018_01_ATL_PHI,75.0,2018,ATL,PHI,pass,M.Ryan,J.Jones,None,10.0,8.0,0.850118,1.0,0.0,right
4,2018_01_ATL_PHI,104.0,2018,ATL,PHI,run,None,None,J.Jones,11.0,NaN,1.005722,0.0,0.0,None
5,2018_01_ATL_PHI,125.0,2018,ATL,PHI,run,None,None,D.Freeman,20.0,NaN,1.478382,0.0,0.0,None
6,2018_01_ATL_PHI,146.0,2018,ATL,PHI,pass,M.Ryan,C.Ridley,None,0.0,4.0,-0.590785,0.0,0.0,right
7,2018_01_ATL_PHI,168.0,2018,ATL,PHI,pass,M.Ryan,D.Freeman,None,0.0,-3.0,-0.877924,0.0,0.0,left


### **La necesidad de cruzar datos con el Roster Histórico**

La hipótesis central es que los Chiefs dejaron de depender de la posición de **Wide Receiver (WR)**. Sin embargo, el dataset de jugadas `df_plays_clean` solo nos indica el nombre del jugador que recibió el balón, pero *no nos dice en qué posición juegan*.

Si no resolvemos esto, no podremos distinguir entre un pase a un receptor abierto muy costoso y un pase a un ala cerrada (*Tight End* - TE) o un corredor (*Running Back* - RB).

Para solucionarlo, descargaremos las bases de datos de los rosters oficiales de esas temporadas y cruzaremos la información para etiquetar cada pase con la posición real del receptor.

In [10]:
df_rosters = nfl.import_seasonal_rosters(years) # Sólo seleccionamos de las temporadas que elegimos
df_rosters.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18503 entries, 0 to 3088
Data columns (total 37 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   season                   18503 non-null  int32         
 1   team                     18503 non-null  object        
 2   position                 18503 non-null  object        
 3   depth_chart_position     18503 non-null  object        
 4   jersey_number            18426 non-null  float64       
 5   status                   18503 non-null  object        
 6   player_name              18503 non-null  object        
 7   first_name               18503 non-null  object        
 8   last_name                18503 non-null  object        
 9   birth_date               16975 non-null  datetime64[ns]
 10  height                   18500 non-null  float64       
 11  weight                   18501 non-null  float64       
 12  college                  18493 non-nul

In [42]:
df_rosters.head(3)

,season,team,position,depth_chart_position,jersey_number,status,player_name,first_name,last_name,birth_date,...,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number,age
0,2018,ARI,K,K,4.0,RES,Phil Dawson,Phil,Dawson,1975-01-23,...,None,Phil,DAW705989,23860,32004441-5770-5989-ac23-bf6cdafcb988,1998.0,1998.0,None,NaN,43.0
1,2018,IND,K,K,4.0,ACT,Adam Vinatieri,Adam,Vinatieri,1972-12-28,...,A01,Adam,VIN196019,21213,32005649-4e19-6019-e626-0b58f9aa81e1,1996.0,1996.0,None,NaN,45.0
2,2018,NE,QB,QB,12.0,ACT,Tom Brady,Tom,Brady,1977-08-03,...,A01,Tom,BRA371156,25511,32004252-4137-1156-7ed0-8b9e44948f13,2000.0,2000.0,NE,199.0,41.0


Vemos que también es un dataset muy completo con hasta **36 variables** como: *peso, fecha de nacimiento, nombre, universidad de la que vienen, etc*. Realmente sólo nos interesa: La temporada, el equipo, el nombre del jugador y la posición.

In [12]:
# Filtramos por las variables que mencionamos
df_rosters_clean = df_rosters[['season', 'team', 'player_name', 'position']]
df_rosters_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18503 entries, 0 to 3088
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   season       18503 non-null  int32 
 1   team         18503 non-null  object
 2   player_name  18503 non-null  object
 3   position     18503 non-null  object
dtypes: int32(1), object(3)
memory usage: 650.5+ KB


Sin valores nulos.

Finalmente guardamos estos nuevos datasets en la carpeta ***/raw*** de nuestro proyecto para tener esta información ya filtrada guardada en un archivo *".csv"*.

In [13]:
df_plays_clean.to_csv('../data/raw/pbp_plays_2018_2023.csv', index=False)
df_rosters_clean.to_csv('../data/raw/rosters_2018_2023.csv', index=False)

## **2. Diagnóstico y Limpieza de Datos**

Antes de cruzar nuestra base de jugadas con los *Rosters*, debemos asegurar la integridad de los datos. Los datos nulos que identificamos anteriormente se deben a:

* **`air_yards` nulos:** Ocurren lógicamente en jugadas por tierra. Debemos imputarlos con `0`.

* **`epa` nulo:** Si la jugada no tiene un valor de *Expected Points Added*, no nos sirve. Procederemos a eliminar esas filas.

* **Jugadores nulos:** Pases lanzados fuera del campo (*throwaways*) o pases bateados en la línea a menudo no registran un receptor.

Pasemos entonces primero a analizar las variables en donde tenemos valores nulos.

In [14]:
df_plays = pd.read_csv('../data/raw/pbp_plays_2018_2023.csv')
print(df_plays.isnull().sum())

game_id                      0
play_id                      0
season                       0
posteam                      0
defteam                      0
play_type                    0
passer_player_name       85796
receiver_player_name     97753
rusher_player_name      122043
yards_gained                 0
air_yards                94358
epa                          1
complete_pass                0
touchdown                    0
pass_location            94364
dtype: int64


Lo primero es rellenar las yardas aéreas con 0 en caso de acarreos.

In [15]:
df_plays['air_yards'] = df_plays['air_yards'].fillna(0)

Después eliminamos jugadas sin métrica de eficiencia (EPA)

In [16]:
df_plays = df_plays.dropna(subset=['epa'])

Ahora crearemos una nueva columna que unifique el jugador objetivo para medir la entropía. La lógica es que si la jugada es pase, el objetivo será el receptor; de lo contrario, en caso de una carrera, el objetivo será el corredor.

In [17]:
df_plays['target_player'] = df_plays['receiver_player_name'].fillna(df_plays['rusher_player_name'])

Por último borramos las jugadas donde no hubo un jugador objetivo claro (*Balores fuera de la cancha, spikes, fumbles en el snap, etc.*).

In [18]:
df_plays = df_plays.dropna(subset=['target_player'])

Veamos ahora cómo quedó nuestro dataset después de esta limpieza.

In [19]:
print(df_plays.isnull().sum())

game_id                      0
play_id                      0
season                       0
posteam                      0
defteam                      0
play_type                    0
passer_player_name       85796
receiver_player_name     85796
rusher_player_name      110085
yards_gained                 0
air_yards                    0
epa                          0
complete_pass                0
touchdown                    0
pass_location            86346
target_player                0
dtype: int64


In [20]:
df_plays.info()

<class 'pandas.core.frame.DataFrame'>
Index: 195881 entries, 0 to 207838
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   game_id               195881 non-null  object 
 1   play_id               195881 non-null  float64
 2   season                195881 non-null  int64  
 3   posteam               195881 non-null  object 
 4   defteam               195881 non-null  object 
 5   play_type             195881 non-null  object 
 6   passer_player_name    110085 non-null  object 
 7   receiver_player_name  110085 non-null  object 
 8   rusher_player_name    85796 non-null   object 
 9   yards_gained          195881 non-null  float64
 10  air_yards             195881 non-null  float64
 11  epa                   195881 non-null  float64
 12  complete_pass         195881 non-null  float64
 13  touchdown             195881 non-null  float64
 14  pass_location         109535 non-null  object 
 15  targe

Vemos que ahora nuestro *DataFrame* está mucho más limpio, únicamente nos quedan valores nulos en 3 columnas que están relacionadas directamente. La razón de esos nulos es que **tuvimos 85,796 jugadas por tierra y 110,085 por aire**, que en conjunto nos dan . Para esto contaremos el total de jugadas por categoría (*run* o *pass*).

In [21]:
df_plays["play_type"].value_counts()

play_type
pass    110085
run      85796
Name: count, dtype: int64

De aquí salen nuestros valores nulos. Por ahora los dejaremos así ya que no afectan directamente a nuestro análisis principal, más adelante únicamente trabajremos con la categoría de "*pass*". Por lo que procedamos a guardar esta versión limpia.

In [22]:
df_plays.to_csv('../data/processed/pbp_plays_clean.csv', index=False)

## **3. Merge entre datasets**
Si bien ahora tenemos ambos dataset limpios y filtrados, hay un pequeño problema para hacer el merge. En el dataset de rosters viene el nombre completo del jugador, mientras que en df_plays el nombre del jugador viene abreviado con el formato: ***PrimerLetraNombre* + *"."* + *Apellido***. Como se ve a continuación

In [23]:
df_rosters_clean.head()

,season,team,player_name,position
0,2018,ARI,Phil Dawson,K
1,2018,IND,Adam Vinatieri,K
2,2018,NE,Tom Brady,QB
3,2018,SEA,Sebastian Janikowski,K
4,2018,HOU,Shane Lechler,P


In [24]:
df_plays['target_player'].head()

0      J.Jones
1      J.Jones
2    D.Freeman
3     C.Ridley
4    D.Freeman
Name: target_player, dtype: object

Lo que haremos será utilizar una función para estandarizar los nombres del roster.

In [ ]:
def estandarizar_nombre(nombre_completo):
    try:
        partes = str(nombre_completo).split(' ', 1) # Dividimos por el primer espacio
        if len(partes) > 1:
            return f"{partes[0][0]}.{partes[1]}" # Aplicamos el formato
        return nombre_completo
    except:
        return nombre_completo

Creamos una nueva columna aplicando la función que acabamos de crear sobre `player_name`, y será esta la que nos servirá como llave para realizar el merge.

In [26]:
df_rosters_clean['target_player'] = df_rosters_clean['player_name'].apply(estandarizar_nombre)

C:\Users\lgram\AppData\Local\Temp\ipykernel_21572\4279150699.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rosters_clean['target_player'] = df_rosters_clean['player_name'].apply(estandarizar_nombre)


#### **Limpieza del Roster antes del cruce**

Eliminamos posibles duplicados, como por ejemplo, si un jugador fue cambiado de equipo a mitad de temporada; y únicamente nos quedamos solo con la posición oficial.

In [27]:
df_rosters_unique = df_rosters_clean.drop_duplicates(subset=['season', 'team', 'target_player'])

### **El Cruce de Tablas (Merge)**

In [28]:
df_final = pd.merge(
    df_plays,
    df_rosters_unique[['season', 'team', 'target_player', 'position']],
    left_on=['season', 'posteam', 'target_player'], # Llaves de la tabla de jugadas
    right_on=['season', 'team', 'target_player'],   # Llaves de la tabla de rosters
    how='left' # Mantenemos todas las jugadas, y traemos la posición si hace match
)

df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195881 entries, 0 to 195880
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   game_id               195881 non-null  object 
 1   play_id               195881 non-null  float64
 2   season                195881 non-null  int64  
 3   posteam               195881 non-null  object 
 4   defteam               195881 non-null  object 
 5   play_type             195881 non-null  object 
 6   passer_player_name    110085 non-null  object 
 7   receiver_player_name  110085 non-null  object 
 8   rusher_player_name    85796 non-null   object 
 9   yards_gained          195881 non-null  float64
 10  air_yards             195881 non-null  float64
 11  epa                   195881 non-null  float64
 12  complete_pass         195881 non-null  float64
 13  touchdown             195881 non-null  float64
 14  pass_location         109535 non-null  object 
 15  

Eliminamos la columna "team" que termina duplicándose.

In [29]:
df_final = df_final.drop(columns=['team'])

### **Post-cruce**
Evaluamos el nuevo dataset.

In [30]:
print(f"Dimensión del dataset final: {df_final.shape}")
print("\nDistribución de las posiciones de los jugadores objetivo:")
print(df_final['position'].value_counts().head(5))

Dimensión del dataset final: (195881, 17)

Distribución de las posiciones de los jugadores objetivo:
position
RB    87612
WR    64584
TE    22650
QB    10822
DB      628
Name: count, dtype: int64


In [31]:
df_final.head()

,game_id,play_id,season,posteam,defteam,play_type,passer_player_name,receiver_player_name,rusher_player_name,yards_gained,air_yards,epa,complete_pass,touchdown,pass_location,target_player,position
0,2018_01_ATL_PHI,75.0,2018,ATL,PHI,pass,M.Ryan,J.Jones,NaN,10.0,8.0,0.850118,1.0,0.0,right,J.Jones,WR
1,2018_01_ATL_PHI,104.0,2018,ATL,PHI,run,NaN,NaN,J.Jones,11.0,0.0,1.005722,0.0,0.0,NaN,J.Jones,WR
2,2018_01_ATL_PHI,125.0,2018,ATL,PHI,run,NaN,NaN,D.Freeman,20.0,0.0,1.478382,0.0,0.0,NaN,D.Freeman,RB
3,2018_01_ATL_PHI,146.0,2018,ATL,PHI,pass,M.Ryan,C.Ridley,NaN,0.0,4.0,-0.590785,0.0,0.0,right,C.Ridley,WR
4,2018_01_ATL_PHI,168.0,2018,ATL,PHI,pass,M.Ryan,D.Freeman,NaN,0.0,-3.0,-0.877924,0.0,0.0,left,D.Freeman,RB


In [32]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195881 entries, 0 to 195880
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   game_id               195881 non-null  object 
 1   play_id               195881 non-null  float64
 2   season                195881 non-null  int64  
 3   posteam               195881 non-null  object 
 4   defteam               195881 non-null  object 
 5   play_type             195881 non-null  object 
 6   passer_player_name    110085 non-null  object 
 7   receiver_player_name  110085 non-null  object 
 8   rusher_player_name    85796 non-null   object 
 9   yards_gained          195881 non-null  float64
 10  air_yards             195881 non-null  float64
 11  epa                   195881 non-null  float64
 12  complete_pass         195881 non-null  float64
 13  touchdown             195881 non-null  float64
 14  pass_location         109535 non-null  object 
 15  

## **Archivos finales**

Finalmente, guardamos la versión final que usaremos.

In [33]:
df_final.to_csv('../data/processed/nfl_storytelling_final.csv', index=False)

## **Datasets extras para la narrativa**

#### **Salary Cap 2022**

Requeríamos datos del salary cap de cada equipo, como no se encontraron datasets con los datos que requeríamos, hicimos el nuestro directamente en excel copiando los datos directamente de las tablas en ***Spotrac***, creando así el archivo *"cap_data_2022_spotrac.csv"* de la carpeta raw.

In [34]:
df_cap_spotrac = pd.read_csv('../data/raw/cap_data_2022_spotrac.csv')
df_cap_spotrac.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Rank       32 non-null     int64  
 1   Equipo     32 non-null     object 
 2   Cap Total  32 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 900.0+ bytes


Como fue un archivo hecho por nosotros, realmente no necesita tratamiento, lo único será cambiar el nombre de la columna "*Equipo*" para estandarizarlo con los demás archivos y utilizar el formato: ***"41.8M"*** para el `Cap Total`.

In [35]:
df_cap_spotrac['Gasto_WR_Millones'] = df_cap_spotrac['Cap Total'] / 1000000
df_cap_spotrac = df_cap_spotrac.rename(columns={'Equipo': 'posteam'})

In [36]:
df_cap_spotrac.head()

,Rank,posteam,Cap Total,Gasto_WR_Millones
0,1,LAR,41473553.0,41.473553
1,2,NE,38554648.0,38.554648
2,3,LAC,38096894.0,38.096894
3,4,NYG,36887201.0,36.887201
4,5,MIA,30138777.0,30.138777


In [43]:
df_cap_spotrac.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rank               32 non-null     int64  
 1   posteam            32 non-null     object 
 2   Cap Total          32 non-null     float64
 3   Gasto_WR_Millones  32 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 1.1+ KB


Ahora sí guardamos el *.csv* en la carpeta de **processed**.

In [37]:
df_cap_spotrac.to_csv('../data/processed/cap_data_2022.csv', index=False)

#### **Ranking de gasto en WR por campeón**
Nuevamente como no tenemos un dataset con los rankings de cada equipo en inversión por posición, lo que hicimos fue hacerlo a mano con los datos rescatados de *Spotrac* año por año. 

Lo único fue busar los campeones del SuperBowl para cada año, y ligar los datos manualmente. Al ser un dataframe específico para el último gráfico, realmente podemos rescatar únicamente la información necesaria: *Temporada, Campeón del Super Bowl, Ranking de gasto en WR y el URL del logo del equipo para un mejor estilo*.

In [38]:
datos_historicos = pd.DataFrame({
    'Temporada': [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'Campeón': [
        'NE Patriots', 'DEN Broncos', 'NE Patriots', 'PHI Eagles', 
        'NE Patriots', 'KC Chiefs', 'TB Buccaneers', 'LA Rams', 
        'KC Chiefs', 'KC Chiefs'
    ],
    'Ranking': [18, 4, 13, 12, 25, 11, 26, 17, 25, 26],
    'URL_Logo': [
        'https://a.espncdn.com/i/teamlogos/nfl/500/ne.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/den.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/ne.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/phi.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/ne.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/kc.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/tb.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/lar.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/kc.png',
        'https://a.espncdn.com/i/teamlogos/nfl/500/kc.png'
    ]
})

Agregamos `Campeón_Label` para el estilo de la etiqueta del hover que tenemos configurada dentro del archivo *app.py*. 

In [39]:
datos_historicos['Campeón_Label'] = '<b>' + datos_historicos['Campeón'] + '</b>'
datos_historicos.head()

,Temporada,Campeón,Ranking,URL_Logo,Campeón_Label
0,2014,NE Patriots,18,https://a.espncdn.com/i/teamlogos/nfl/500/ne.png,<b>NE Patriots</b>
1,2015,DEN Broncos,4,https://a.espncdn.com/i/teamlogos/nfl/500/den.png,<b>DEN Broncos</b>
2,2016,NE Patriots,13,https://a.espncdn.com/i/teamlogos/nfl/500/ne.png,<b>NE Patriots</b>
3,2017,PHI Eagles,12,https://a.espncdn.com/i/teamlogos/nfl/500/phi.png,<b>PHI Eagles</b>
4,2018,NE Patriots,25,https://a.espncdn.com/i/teamlogos/nfl/500/ne.png,<b>NE Patriots</b>


Guardamos ahora sí el *.csv* final.

In [40]:
datos_historicos.to_csv('../data/processed/historico_superbowl_wr.csv', index=False)